# PTR-MS Data Analysis

Analysis of Ionikon PTR-MS HDF5 data files using `analyze_ptrms.py`.

Edit `DATA_FILES` below to point to your file(s).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Import analysis module from same directory
import sys
sys.path.insert(0, str(Path().resolve()))
from analyze_ptrms import (
    read_file, instrument_summary, build_peak_list,
    get_traces, calibrated_mz,
)

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## Configuration

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
DATA_FILES = sorted(Path('.').glob('Data*.h5'))   # all Data*.h5 in current dir
# DATA_FILES = [Path('my_specific_file.h5')]       # or point to a specific file

MIN_CONC_PPB   = 0.1     # minimum mean concentration to include in peak list
TOP_N_PEAKS    = 30      # number of peaks to show in bar chart
MZ_RANGE       = (20, 200)   # m/z range for average spectrum plot

# Compounds to highlight in time-series (substring match against formula names)
COMPOUNDS_OF_INTEREST = ['CH4O', 'C2H6O', 'HCl', 'CH2O', 'C3H6O']
# ─────────────────────────────────────────────────────────────────────────────

print(f'Found {len(DATA_FILES)} file(s):')
for f in DATA_FILES:
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')

## Load Data

In [ ]:
datasets = {}
for path in DATA_FILES:
    print(f'Loading {path.name} …', end=' ', flush=True)
    datasets[path] = read_file(path)
    print('done')

# Work with the first file by default for single-file sections
path0 = DATA_FILES[0]
data  = datasets[path0]

## Instrument Summary

In [ ]:
for path, d in datasets.items():
    s = instrument_summary(d)
    print(f'─── {path.name} ───')
    print(f"  Start time  : {s['start_time']}")
    print(f"  Duration    : {s['duration_min']} min  ({s['n_spectra']} spectra)")
    print(f"  Traced masses: {s['n_masses']}")
    print(f"  E/N         : {s.get('E/N_Act [Td]', 'n/a')} Td")
    print(f"  Drift V     : {s.get('Udrift_Act [V]', 'n/a')} V")
    print(f"  Drift p     : {s.get('p-Drift_Act [mbar]', 'n/a')} mbar")
    print(f"  Drift T     : {s.get('T-Drift_Act [°C]', 'n/a')} °C")
    print(f"  H3O+ purity : {s.get('H3O+ [%]', 'n/a')} %")
    print(f"  H2O cluster : {s.get('H2O.H3O+ (Cluster) [%]', 'n/a')} %")
    print(f"  O2+         : {s.get('O2+ [%]', 'n/a')} %")
    print(f"  PI total    : {s.get('PI (total) [x1E6]', 'n/a')} ×10⁶ cps")
    print()

### Primary ion composition over time

In [ ]:
fig, axes = plt.subplots(len(datasets), 1,
                         figsize=(12, 3.5 * len(datasets)),
                         squeeze=False)

pi_cols = ['H3O+ [%]', 'H2O.H3O+ (Cluster) [%]', 'O2+ [%]', 'NO+ [%]']
colors  = ['steelblue', 'darkorange', 'seagreen', 'firebrick']

for ax, (path, d) in zip(axes[:, 0], datasets.items()):
    t = (d['timestamps'] - d['timestamps'][0]).total_seconds() / 60
    for col, c in zip(pi_cols, colors):
        if col in d['calc'].columns:
            ax.plot(t, d['calc'][col], label=col.replace(' [%]', ''), color=c)
    ax.set_ylabel('Fraction (%)')
    ax.set_title(f'Primary ion composition — {path.name}')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_xlabel('Elapsed time (min)')

fig.tight_layout()
plt.show()

## Peak List

In [ ]:
peak_lists = {}
for path, d in datasets.items():
    peaks = build_peak_list(d, min_conc=MIN_CONC_PPB, exclude_primary=True)
    peak_lists[path] = peaks
    print(f'{path.name}: {len(peaks)} compounds above {MIN_CONC_PPB} ppbv')

# Display for the first file
peaks = peak_lists[path0]
peaks[['name', 'mz', 'mean_ppbv', 'std_ppbv', 'min_ppbv', 'max_ppbv', 'mean_raw_cps']] \
    .style \
    .format({'mz': '{:.4f}', 'mean_ppbv': '{:.3f}', 'std_ppbv': '{:.3f}',
             'min_ppbv': '{:.3f}', 'max_ppbv': '{:.3f}', 'mean_raw_cps': '{:.0f}'}) \
    .background_gradient(subset=['mean_ppbv'], cmap='YlOrRd') \
    .set_caption(f'Peak list — {path0.name}')

### Peak list bar chart

In [ ]:
for path, peaks in peak_lists.items():
    top = peaks.head(TOP_N_PEAKS)
    x   = np.arange(len(top))

    fig, ax = plt.subplots(figsize=(14, 5))
    bars = ax.bar(x, top['mean_ppbv'], yerr=top['std_ppbv'],
                  capsize=3, color='steelblue', alpha=0.8, ecolor='navy')

    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{r['name']}\n{r['mz']:.3f}" for _, r in top.iterrows()],
        rotation=45, ha='right', fontsize=8,
    )
    ax.set_ylabel('Mean concentration (ppbv)')
    ax.set_yscale('log')
    ax.set_title(f'Top {TOP_N_PEAKS} compounds — {path.name}')
    fig.tight_layout()
    plt.show()

## Average Mass Spectrum

In [ ]:
for path, d in datasets.items():
    spec = d['avg_spectrum']
    mz   = calibrated_mz(len(spec), d['cal_mapping'])
    mask = (mz >= MZ_RANGE[0]) & (mz <= MZ_RANGE[1])

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(mz[mask], spec[mask], linewidth=0.6, color='steelblue')

    # Annotate traced peaks within the m/z range
    peaks_in_range = peak_lists[path][
        (peak_lists[path]['mz'] >= MZ_RANGE[0]) &
        (peak_lists[path]['mz'] <= MZ_RANGE[1]) &
        (peak_lists[path]['mean_ppbv'] >= 1.0)
    ]
    for _, row in peaks_in_range.iterrows():
        # Find nearest spectrum bin to annotate
        idx = np.argmin(np.abs(mz[mask] - row['mz']))
        mz_ann = mz[mask][idx]
        sig_ann = spec[mask][idx]
        ax.annotate(
            f"{row['name']}\n{row['mz']:.3f}",
            xy=(mz_ann, sig_ann),
            xytext=(0, 18), textcoords='offset points',
            ha='center', fontsize=7, color='firebrick',
            arrowprops=dict(arrowstyle='->', color='firebrick', lw=0.8),
        )

    ax.set_xlabel('m/z')
    ax.set_ylabel('Signal (counts)')
    ax.set_xlim(MZ_RANGE)
    ax.set_title(f'Average spectrum (m/z {MZ_RANGE[0]}–{MZ_RANGE[1]}) — {path.name}')
    fig.tight_layout()
    plt.show()

## Time Series — Compounds of Interest

In [ ]:
for path, d in datasets.items():
    traces = get_traces(d, COMPOUNDS_OF_INTEREST)
    if traces.empty:
        print(f'{path.name}: no matching compounds found')
        continue

    elapsed = (traces.index - traces.index[0]).total_seconds() / 60

    fig, ax = plt.subplots(figsize=(13, 5))
    for col in traces.columns:
        ax.plot(elapsed, traces[col], label=col, linewidth=1.4)

    ax.set_xlabel('Elapsed time (min)')
    ax.set_ylabel('Concentration (ppbv)')
    ax.set_title(f'Time series — {path.name}')
    ax.legend(fontsize=8, ncol=2, loc='upper right')
    fig.tight_layout()
    plt.show()

## Reaction Conditions Over Time

In [ ]:
for path, d in datasets.items():
    rxn = d['rxn']
    elapsed = (rxn.index - rxn.index[0]).total_seconds() / 60
    cols = [c for c in rxn.columns if any(k in c for k in
            ['Udrift', 'p-Drift', 'T-Drift', 'E/N'])]

    if not cols:
        continue

    fig, axes = plt.subplots(len(cols), 1,
                             figsize=(12, 2.5 * len(cols)),
                             sharex=True, squeeze=False)
    for ax, col in zip(axes[:, 0], cols):
        ax.plot(elapsed, rxn[col], color='steelblue', linewidth=1.2)
        ax.set_ylabel(col, fontsize=9)

    axes[-1, 0].set_xlabel('Elapsed time (min)')
    fig.suptitle(f'Reaction conditions — {path.name}', y=1.01)
    fig.tight_layout()
    plt.show()

## Multi-file Comparison

Only meaningful when more than one file is loaded.

In [ ]:
if len(datasets) > 1:
    # Merge peak lists on compound name, compare mean concentrations
    frames = []
    for path, peaks in peak_lists.items():
        tmp = peaks[['name', 'mz', 'mean_ppbv']].copy()
        tmp = tmp.rename(columns={'mean_ppbv': path.stem})
        frames.append(tmp.set_index('name'))

    merged = frames[0]
    for f in frames[1:]:
        merged = merged.join(f.drop(columns='mz', errors='ignore'), how='outer')
    merged = merged.fillna(0).sort_values(merged.columns[1], ascending=False)

    top = merged.head(20)
    x = np.arange(len(top))
    width = 0.8 / len(datasets)
    fig, ax = plt.subplots(figsize=(14, 6))
    for i, col in enumerate(merged.columns[1:]):
        ax.bar(x + i * width - 0.4 + width / 2, top[col], width, label=col, alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(top.index, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Mean concentration (ppbv)')
    ax.set_yscale('log')
    ax.set_title('Multi-file comparison — top 20 compounds')
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print('Load multiple files via DATA_FILES to see this comparison.')

## Export Peak Lists to CSV

In [ ]:
for path, peaks in peak_lists.items():
    out = path.parent / f'{path.stem}_peak_list.csv'
    peaks.to_csv(out, index=False)
    print(f'Saved: {out}')

if len(peak_lists) > 1:
    combined = pd.concat(
        [p.assign(file=path.name) for path, p in peak_lists.items()],
        ignore_index=True,
    )
    out = DATA_FILES[0].parent / 'combined_peak_list.csv'
    combined.to_csv(out, index=False)
    print(f'Combined saved: {out}')